# CONSUMER_REVIEW table
Written by: Capleton

This notebook makes a CSV file that fits the table consumer_review in our PostgreSQL database.

## STEP 1: import os and pandas
We need pandas to prepare the data before it goes into the database, and os to save the file for the import in pgAdmin.

In [1]:
import os
import pandas as pd

## STEP 2: read the files
We read the cleaned user reviews and the movie table. The movie table is needed because every review must get the movie_id of its movie. Before we change anything, we look at the first rows and count the empty values, so we know what goes into the database. We also check that movie has a unique movie_id, otherwise a review could be linked to two movies.

In [2]:
folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\clean"

User_reviews = pd.read_csv(os.path.join(folder, "UserReviews_clean.csv"), low_memory=False)
movie = pd.read_csv(os.path.join(folder, "movie.csv"))

print("User reviews:", len(User_reviews))
print("Movies:", len(movie))

print(User_reviews[["url", "idvscore", "posemo", "negemo", "thumbsUp", "thumbsTot"]].head())
print(User_reviews[["url", "idvscore", "posemo", "negemo", "thumbsUp", "thumbsTot"]].isnull().sum())

print("Check, movie has movie_id and url:", "movie_id" in movie.columns and "url" in movie.columns)
print("Check, movie_id is unique:", movie["movie_id"].is_unique)

User reviews: 319662
Movies: 11332
                                        url idvscore  posemo  negemo thumbsUp  \
0  https://www.metacritic.com/movie/bronson        8    0.00    0.88        2   
1  https://www.metacritic.com/movie/bronson        9    2.91    0.97        0   
2  https://www.metacritic.com/movie/bronson       10    4.20    0.70        1   
3  https://www.metacritic.com/movie/bronson        6   14.29    0.00        0   
4  https://www.metacritic.com/movie/bronson        8    5.00    0.00        0   

  thumbsTot  
0         2  
1         1  
2         1  
3         1  
4         0  
url             0
idvscore     3404
posemo          0
negemo          0
thumbsUp     3580
thumbsTot    3576
dtype: int64
Check, movie has movie_id and url: True
Check, movie_id is unique: True


## STEP 3: keep only the ERD columns and give them the ERD names
The review file has around 90 columns, but our ERD only uses the columns that answer our research questions: the score, the positive and negative emotion, and the thumbs that show how useful other people found the review. The names must be the same as in the table in PostgreSQL, otherwise pgAdmin does not know which value goes into which column.

Some cells in the score and thumbs columns contain text instead of a number. PostgreSQL only accepts numbers in a FLOAT or INT column, so to_numeric turns every number into a real number and every text into an empty value (NULL).

In [3]:
User_reviews = User_reviews.rename(columns={"idvscore": "review_score",  "posemo": "pos_emotion_pct",  "negemo": "neg_emotion_pct",  "thumbsUp": "thumbs_up", "thumbsTot": "total_thumbs"})

User_reviews = User_reviews[["url", "review_score", "pos_emotion_pct", "neg_emotion_pct", "thumbs_up", "total_thumbs"]]

User_reviews["review_score"] = pd.to_numeric(User_reviews["review_score"], errors="coerce")
User_reviews["thumbs_up"] = pd.to_numeric(User_reviews["thumbs_up"], errors="coerce").astype("Int64")
User_reviews["total_thumbs"] = pd.to_numeric(User_reviews["total_thumbs"], errors="coerce").astype("Int64")

print(list(User_reviews.columns))
print(User_reviews[["review_score", "thumbs_up", "total_thumbs"]].isnull().sum())

['url', 'review_score', 'pos_emotion_pct', 'neg_emotion_pct', 'thumbs_up', 'total_thumbs']
review_score    3409
thumbs_up       3581
total_thumbs    3582
dtype: int64


## STEP 4: fill movie_id with the url
movie_id is the foreign key to MOVIE. The consumer reviews and the movie table come from the same Metacritic data, so they share the url. The url is unique per movie, so we do not need a match key here: we look up the url in MOVIE and take the movie_id. Spaces at the start and end of the url are removed first, so the same url is written the same way in both files. The check makes sure the number of reviews stays the same: if it grows, a url was found twice in movie.

In [4]:
User_reviews_before = len(User_reviews)
User_reviews["url"] = User_reviews["url"].str.strip()
movie["url"] = movie["url"].str.strip()

User_reviews = User_reviews.merge(movie[["movie_id", "url"]], on="url", how="left")

print("User reviews with a movie:", User_reviews["movie_id"].notna().sum())
print("User reviews without a movie:", User_reviews["movie_id"].isna().sum())
print("Check, no review got two movies:", len(User_reviews) == User_reviews_before)

User reviews with a movie: 319573
User reviews without a movie: 89
Check, no review got two movies: True


## STEP 5: remove reviews without a movie
movie_id is NOT NULL in consumer_review, and a foreign key must point to a movie that exists in MOVIE. A review of a movie that is not in our database cannot be imported, so it is removed. Int64 keeps movie_id a whole number, because PostgreSQL does not accept 12.0 in an INT column.

In [5]:
User_reviews = User_reviews.dropna(subset=["movie_id"])
User_reviews["movie_id"] = User_reviews["movie_id"].astype("Int64")

print("User reviews left:", len(User_reviews))
print("Check, no empty movie_id left:", User_reviews["movie_id"].isna().sum() == 0)

User reviews left: 319573
Check, no empty movie_id left: True


## STEP 6: give every review a review_id
review_id is the primary key. The database needs it to tell every review apart, so it must be unique and can never be empty. We make it in Python, the same as all other IDs in our database.

In [6]:
User_reviews = User_reviews.reset_index(drop=True)
User_reviews["review_id"] = User_reviews.index + 1

print("Check, review_id is unique:", User_reviews["review_id"].is_unique)
print(User_reviews.head())

Check, review_id is unique: True
                                        url  review_score  pos_emotion_pct  \
0  https://www.metacritic.com/movie/bronson           8.0             0.00   
1  https://www.metacritic.com/movie/bronson           9.0             2.91   
2  https://www.metacritic.com/movie/bronson          10.0             4.20   
3  https://www.metacritic.com/movie/bronson           6.0            14.29   
4  https://www.metacritic.com/movie/bronson           8.0             5.00   

   neg_emotion_pct  thumbs_up  total_thumbs  movie_id  review_id  
0             0.88          2             2      1565          1  
1             0.97          0             1      1565          2  
2             0.70          1             1      1565          3  
3             0.00          0             1      1565          4  
4             0.00          0             0      1565          5  


## STEP 7: put the columns in the same order as the table
The order of the columns in the CSV must be the same as in the table in PostgreSQL: first the primary key, then the foreign key, then the rest. consumer_review has 8 columns, so the CSV must have the same 8 columns, including thumbs_up and total_thumbs.

In [7]:
User_reviews = User_reviews[["review_id", "movie_id", "url", "review_score", "pos_emotion_pct",
                             "neg_emotion_pct", "thumbs_up", "total_thumbs"]]

print(User_reviews.head())
print("Check, primary key is first:", User_reviews.columns[0] == "review_id")
print("Check, 8 columns:", len(User_reviews.columns) == 8)

   review_id  movie_id                                       url  \
0          1      1565  https://www.metacritic.com/movie/bronson   
1          2      1565  https://www.metacritic.com/movie/bronson   
2          3      1565  https://www.metacritic.com/movie/bronson   
3          4      1565  https://www.metacritic.com/movie/bronson   
4          5      1565  https://www.metacritic.com/movie/bronson   

   review_score  pos_emotion_pct  neg_emotion_pct  thumbs_up  total_thumbs  
0           8.0             0.00             0.88          2             2  
1           9.0             2.91             0.97          0             1  
2          10.0             4.20             0.70          1             1  
3           6.0            14.29             0.00          0             1  
4           8.0             5.00             0.00          0             0  
Check, primary key is first: True
Check, 8 columns: True


## STEP 8: save as CSV
This file is imported into consumer_review in pgAdmin. index=False prevents an extra column with row numbers, because that column does not exist in the table. At the end we read the file back, to check that everything was saved.

In [13]:
new_file = os.path.join(folder, "consumer_review.csv")

User_reviews.to_csv(new_file, index=False)

print("Saved:", new_file)

check = pd.read_csv(new_file)
print("Check, same number of rows in the file:", len(check) == len(User_reviews))
print("Check, 8 columns:", len(check.columns) == 8)

Saved: C:\Users\caple\OneDrive - HvA\Master\Data management\clean\consumer_review.csv
Check, same number of rows in the file: True
Check, 8 columns: True
